# 09 – Quantitative comparison of the forecasts

Scores of the forecasts of the PINN, the Mechanistic model and the baseline , against the observations. One notebook for the four synthetic scenarios and for the real case, chosen with `CASE` (as in notebooks 03–08).

Scores (not weighted): **MAE** and **RMSE** of the median, **WIS**, **IS_80** (interval score of the 80% interval), **wMAPE** and the **coverage** of the 50% and 90% intervals. Only the future is scored (horizon h = t − t0 ≥ 1, not the fit), on the windows that all the models have in common. Four tables for every case:

1. per horizon (h = 1, 2, 3, 4);
2. per horizon, errors divided by those of the baseline (relative scores: below 1 is better than the baseline);
3. pooled over the horizons (every (t0, h) is an independent point);
4. pooled, relative to the baseline.

The coverage is never normalised. The observations are the series of the case (weekly deaths, or the noisy weekly infections of the synthetic scenarios), as in the plots of notebook 08, and the weeks scored are the same:

- **synthetic**: up to T_LAST = 33 (R0 = 1.5) or 23 (R0 = 2.0) (`case.t0_last`); only the windows whose whole horizon falls before T_LAST are scored, so every horizon has the same number of windows;
- **real**: windows from 2020-03-15 and targets up to 2020-07-19, so the longer horizons have fewer windows (18, 17, 16, 15 rounds, 66 points).

Input: forecasts of the PINN (notebook 04), of the Mechanistic model (notebook 05) and of the baseline (notebook 06), from `precomputed/` or `outputs/`. Output: `tables/forecasts/<case>/*.csv`.

In [ ]:
import os
import pickle

import numpy as np

from pinnvsmech.evaluation.best_seed import resolve_run_dir
from pinnvsmech.evaluation.metrics import (
    TABLE_COLUMNS, compute_metrics_pooled, compute_metrics_per_horizon, normalize_by_baseline,
    normalize_pooled_by_baseline,
)
from pinnvsmech.forecasting.baseline import baseline_label
from pinnvsmech.forecasting.windows import select_windows
from pinnvsmech.inference.cases import load_case
from pinnvsmech.paths import OUTPUTS_DIR, PRECOMPUTED_DIR, TABLES_DIR

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 1.5                      # synthetic only: 1.5 or 2.0
NOISE = "mild"                     # synthetic only: "light" or "mild"
CASE_OVERRIDES = {}                # synthetic: look-back on all the history (default). Real: look-back of 5 weeks, no
                                   # weights (default); dict(dt_train=None) uses all the history (look-back 25, weighted)

SOURCE = "precomputed"             # "precomputed" (results of the paper) or "outputs" (new runs)
PINN_SEED = "best"                 # PINN run: "best" = folder best_WIS of the case, or the seed chosen in notebook 10; or the seed number (e.g. 10) to use seed_010
HORIZON = 4                        # horizon of the forecast files (weeks)
HORIZONS = (1, 2, 3, 4)            # horizons scored

COMMON_ONLY = True                 # pooled table: only the windows common to all the models
SAVE_TABLES = False

## Data and forecasts

In [ ]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
# The Mechanistic results do not depend on the weights of the likelihood
mech_label = load_case(CASE, R0=R0_TRUE, noise=NOISE,
                       **{k: v for k, v in CASE_OVERRIDES.items() if k != "weight"}).label

root = PRECOMPUTED_DIR if SOURCE == "precomputed" else OUTPUTS_DIR


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


baseline_dir = root / "baselines" / baseline_label(case)
pinn_run = resolve_run_dir(PINN_SEED, root / "posteriors" / "pinn" / case.label)
forecasts = {
    "PINN": load_pickle(pinn_run / f"forecast_h{HORIZON}" / "forecast_results.pickle"),
    "Mechanistic": load_pickle(root / "posteriors" / "mechanistic" / mech_label / f"forecast_h{HORIZON}"
                               / "forecast_results.pickle"),
    "Baseline": load_pickle(baseline_dir / "forecast_results_baseline.pkl"),
}

# Weeks scored: synthetic -> windows whose horizon ends before T_LAST; real -> 2020-03-15 ... 2020-07-19
forecasts, t0_min, t_last = select_windows(case, forecasts, HORIZON)
obs = case.obs
print(case.label, "| PINN run", pinn_run.name, "| last week scored:", t_last, "| first window:", t0_min)
for name, f in forecasts.items():
    t0s = sorted(int(e["t0"]) for e in f.values())
    print(f"  {name:12s} {len(t0s)} windows, t0 {t0s[0]}..{t0s[-1]}")

model_order = [m for m in ("PINN", "Mechanistic", "Baseline") if m in forecasts]

table_dir = TABLES_DIR / "forecasts" / case.label
os.makedirs(table_dir, exist_ok=True)


def show(df, title, float_format="{:.4f}"):
    print("\n" + "=" * 92 + f"\n  {title}\n" + "=" * 92)
    print(df.to_string(index=False, float_format=lambda x: float_format.format(x)))


def save(df, name):
    if SAVE_TABLES:
        df.to_csv(table_dir / f"{name}.csv", index=False)

## Per horizon

In [ ]:
results = compute_metrics_per_horizon(forecasts, obs, horizons=HORIZONS, t0_min=t0_min, t_last=t_last)
results_norm = normalize_by_baseline(results)

for h in HORIZONS:
    df_h = results[results["Horizon"] == h].set_index("Model").reindex(model_order).reset_index()
    show(df_h[["Model"] + TABLE_COLUMNS], f"Horizon {h} - {int(df_h['n_rounds'].iloc[0])} forecast rounds")

for h in HORIZONS:
    df_h = results_norm[results_norm["Horizon"] == h].set_index("Model").reindex(model_order).reset_index()
    show(df_h[["Model"] + TABLE_COLUMNS], f"Horizon {h} - relative to the baseline", "{:.3f}")

save(results, "metrics_per_horizon")
save(results_norm, "metrics_per_horizon_relative_to_baseline")

## Pooled over the horizons

In [ ]:
pooled = compute_metrics_pooled(forecasts, obs, horizons=HORIZONS, common_only=COMMON_ONLY, t0_min=t0_min,
                                t_last=t_last)
pooled = pooled.set_index("Model").reindex(model_order).reset_index()
pooled_norm = normalize_pooled_by_baseline(pooled)

show(pooled[["Model", "n_points"] + TABLE_COLUMNS], "Pooled over the horizons")
show(pooled_norm[["Model"] + TABLE_COLUMNS], "Pooled over the horizons - relative to the baseline")

save(pooled, "metrics_pooled")
save(pooled_norm, "metrics_pooled_relative_to_baseline")